In [ ]:
# %%
# 01 — Fresh root, schema version, and config
from pathlib import Path
import os
import sys
import json
import shutil
import warnings

warnings.filterwarnings("ignore")

print("Python:", sys.version)
print("Executable:", sys.executable)

ROOT = Path.home() / "projects" / "allen_preproc_gratings_lsn_v1"
RESET_ROOT = True

if RESET_ROOT and ROOT.exists():
    shutil.rmtree(ROOT)

CACHE_DIR = ROOT / "allen_cache"
META_DIR = ROOT / "metadata"
EXP_DIR = ROOT / "experiments"
INDEX_DIR = ROOT / "indexes"
LOG_DIR = ROOT / "logs"
BANK_DIR = ROOT / "stimulus_bank"
TRANSPORT_DIR = ROOT / "transport"

for p in [ROOT, CACHE_DIR, META_DIR, EXP_DIR, INDEX_DIR, LOG_DIR, BANK_DIR, TRANSPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CFG = {
    "SCHEMA_VERSION": "gratings_lsn_v1",
    "NOTEBOOK_VERSION": "A_gratings_lsn_v1",

    "MANIFEST_PATH": str(CACHE_DIR / "brain_observatory_manifest.json"),

    "MAX_EXPERIMENTS": None,
    "FILTER_SESSION_TYPES": None,
    "N_WORKERS": min(64, os.cpu_count() or 8),

    "SAVE_FULL_DFF": False,
    "SAVE_FULL_CORRECTED_FLUORESCENCE": False,
    "SAVE_ROI_MASKS": False,
    "SAVE_RUNNING_FULL": True,
    "SAVE_STIMULUS_TABLES_IN_H5": True,
    "SAVE_STIMULUS_TEMPLATES_GLOBAL": True,

    # focus only on gratings + locally sparse noise
    "CANDIDATE_STIMULI": [
        "static_gratings",
        "drifting_gratings",
        "locally_sparse_noise",
        "locally_sparse_noise_4deg",
        "locally_sparse_noise_8deg",
    ],

    # per-presentation summaries only for the stimuli we care about
    "SUMMARY_STIMULI": [
        "static_gratings",
        "drifting_gratings",
        "locally_sparse_noise",
        "locally_sparse_noise_4deg",
        "locally_sparse_noise_8deg",
    ],

    "MAX_PRESENTATIONS_FOR_SUMMARY": 20000,

    "COMPUTE_STATIC_CONDITION_SUMMARY": True,
    "COMPUTE_POOLED_COV_DELTA": True,

    "FLOAT_DTYPE": "float32",
    "COMPRESSION": "gzip",
    "COMPRESSION_OPTS": 4,

    "SEED": 0,
}

with open(LOG_DIR / "config_notebook_A.json", "w") as f:
    json.dump(CFG, f, indent=2)

print(json.dumps(CFG, indent=2))

In [ ]:
# %%
# 02 — Imports and AllenSDK setup
import hashlib
import traceback
import uuid
import tempfile

import numpy as np
import pandas as pd
import h5py

from joblib import Parallel, delayed
from sklearn.covariance import LedoitWolf
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

FLOAT_NP = np.float32 if CFG["FLOAT_DTYPE"] == "float32" else np.float64

boc = BrainObservatoryCache(manifest_file=CFG["MANIFEST_PATH"])
print("BrainObservatoryCache ready.")
print("Manifest:", CFG["MANIFEST_PATH"])

In [ ]:
# %%
# 03 — Metadata table
def normalize_experiment_metadata(records):
    df = pd.DataFrame(records).copy()

    donor_candidates = [
        "donor_name",
        "specimen_name",
        "specimen_id",
        "donor_id",
        "mouse_id",
    ]

    donor_name = []
    for _, row in df.iterrows():
        val = None
        for c in donor_candidates:
            if c in row and pd.notna(row[c]):
                val = row[c]
                break
        donor_name.append(str(val) if val is not None else "unknown")

    df["donor_name"] = donor_name

    keep_cols = [
        c for c in [
            "id",
            "experiment_container_id",
            "session_type",
            "targeted_structure",
            "imaging_depth",
            "cre_line",
            "fail_eye_tracking",
            "donor_name",
        ] if c in df.columns
    ]
    return df[keep_cols].copy()

meta_records = boc.get_ophys_experiments()
meta_df = normalize_experiment_metadata(meta_records)

if CFG["FILTER_SESSION_TYPES"] is not None and "session_type" in meta_df.columns:
    meta_df = meta_df[meta_df["session_type"].isin(CFG["FILTER_SESSION_TYPES"])].copy()

meta_df = meta_df.sort_values(
    ["targeted_structure", "imaging_depth", "id"]
).reset_index(drop=True)

meta_df.to_csv(META_DIR / "all_ophys_experiments_metadata.csv", index=False)

print("Total experiments in metadata:", len(meta_df))
display(meta_df.head())
display(
    meta_df.groupby("targeted_structure")
    .size()
    .rename("n_experiments")
    .reset_index()
    .sort_values("n_experiments", ascending=False)
)

In [ ]:
# %%
# 04 — Paths and metadata helpers
def exp_dir(exp_id: int) -> Path:
    return EXP_DIR / f"ophys_experiment_{int(exp_id)}"

def store_path(exp_id: int) -> Path:
    return exp_dir(exp_id) / "store.h5"

def meta_json_path(exp_id: int) -> Path:
    return exp_dir(exp_id) / "meta.json"

def bank_template_path(stim_name: str, sha1: str) -> Path:
    return BANK_DIR / f"{stim_name}__{sha1}.npz"

def safe_jsonable(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x

def sha1_of_array(arr: np.ndarray) -> str:
    arr_c = np.ascontiguousarray(arr)
    h = hashlib.sha1()
    h.update(str(arr_c.shape).encode("utf-8"))
    h.update(str(arr_c.dtype).encode("utf-8"))
    h.update(arr_c.tobytes())
    return h.hexdigest()

def is_excitatory(cre_line: str) -> bool:
    cre_line = str(cre_line)
    excit_keywords = ["Cux2", "Rorb", "Scnn1a", "Rbp4", "Tlx3", "Ntsr1", "Slc17a7", "Emx1"]
    inhib_keywords = ["Sst", "Vip", "Pvalb", "Gad2", "Ndnf", "Htr3a"]
    if any(k in cre_line for k in inhib_keywords):
        return False
    return any(k in cre_line for k in excit_keywords)

def canonical_visp_layer_label(cre_line: str, imaging_depth):
    cre_line = str(cre_line)
    try:
        imaging_depth = int(imaging_depth)
    except Exception:
        return None

    if "Ntsr1" in cre_line and imaging_depth >= 430:
        return "L6"
    if ("Rbp4" in cre_line or "Tlx3" in cre_line) and 300 <= imaging_depth <= 470:
        return "L5"
    if ("Rorb" in cre_line or "Scnn1a" in cre_line) and 220 <= imaging_depth <= 330:
        return "L4"
    if "Cux2" in cre_line and 100 <= imaging_depth <= 230:
        return "L2/3"
    if ("Slc17a7" in cre_line or "Emx1" in cre_line):
        if 100 <= imaging_depth <= 190:
            return "L2/3"
        if 230 <= imaging_depth <= 300:
            return "L4"
        if 330 <= imaging_depth <= 430:
            return "L5"
        if imaging_depth >= 470:
            return "L6"
    return None

In [ ]:
# %%
# 05 — HDF5 helpers
def write_h5_dataset(group, name, arr):
    group.create_dataset(
        name,
        data=arr,
        compression=CFG["COMPRESSION"],
        compression_opts=CFG["COMPRESSION_OPTS"],
        shuffle=True,
    )

def write_dataframe_to_h5(parent_group, group_name: str, df: pd.DataFrame):
    g = parent_group.create_group(group_name)
    g.attrs["n_rows"] = int(len(df))
    g.attrs["columns"] = json.dumps(list(df.columns))

    str_dtype = h5py.string_dtype(encoding="utf-8")

    for col in df.columns:
        s = df[col]
        isna = s.isna().to_numpy(dtype=np.uint8)

        if pd.api.types.is_numeric_dtype(s):
            arr = s.to_numpy()
            g.create_dataset(
                col,
                data=arr,
                compression=CFG["COMPRESSION"],
                compression_opts=CFG["COMPRESSION_OPTS"],
                shuffle=True,
            )
            g[col].attrs["stored_dtype"] = str(arr.dtype)
            g.create_dataset(f"__isna__{col}", data=isna)

        elif pd.api.types.is_bool_dtype(s):
            arr = s.fillna(False).astype(np.uint8).to_numpy()
            g.create_dataset(col, data=arr)
            g[col].attrs["stored_dtype"] = "bool"
            g.create_dataset(f"__isna__{col}", data=isna)

        else:
            arr = s.fillna("").astype(str).to_numpy(dtype=object)
            g.create_dataset(col, data=arr, dtype=str_dtype)
            g[col].attrs["stored_dtype"] = "string"
            g.create_dataset(f"__isna__{col}", data=isna)

    return g

def read_dataframe_from_h5(group):
    cols = json.loads(group.attrs["columns"])
    out = {}

    for col in cols:
        arr = group[col][:]
        isna = group[f"__isna__{col}"][:].astype(bool)
        stored_dtype = group[col].attrs.get("stored_dtype", "")
        if isinstance(stored_dtype, bytes):
            stored_dtype = stored_dtype.decode("utf-8")

        if stored_dtype == "bool":
            vals = arr.astype(float)
            vals[isna] = np.nan
            out[col] = vals
        elif stored_dtype == "string":
            vals = np.array([
                x.decode("utf-8") if isinstance(x, bytes) else str(x)
                for x in arr
            ], dtype=object)
            vals[isna] = None
            out[col] = vals
        else:
            vals = arr.copy()
            out[col] = vals

    return pd.DataFrame(out)

In [ ]:
# %%
# 06 — Running speed and stimulus helpers
def authoritative_running_speed_from_sdk(ds):
    # AllenSDK source of truth: (speed, time)
    run_speed, run_time = ds.get_running_speed()

    run_speed = np.asarray(run_speed, dtype=np.float64).ravel()
    run_time = np.asarray(run_time, dtype=np.float64).ravel()

    valid = np.isfinite(run_speed) & np.isfinite(run_time)
    run_speed = run_speed[valid]
    run_time = run_time[valid]

    if len(run_time) < 10:
        raise RuntimeError("Too few valid running-speed samples.")

    order = np.argsort(run_time)
    run_time = run_time[order]
    run_speed = run_speed[order]

    # collapse duplicate timestamps
    tmp = pd.DataFrame({"t": run_time, "v": run_speed}).groupby("t", as_index=False)["v"].mean()
    run_time = tmp["t"].to_numpy(dtype=np.float64)
    run_speed = tmp["v"].to_numpy(dtype=np.float64)

    return run_speed, run_time

def compute_trial_running_speed(t_ophys, run_time, run_speed, start_frames, end_frames):
    run_interp = np.interp(t_ophys, run_time, run_speed).astype(FLOAT_NP)

    stim_speed = []
    pre_speed = []

    for s, e in zip(start_frames, end_frames):
        s = int(s)
        e = int(max(e, s + 1))

        stim_speed.append(float(np.nanmean(run_interp[s:e])))

        pre_len = e - s
        b0 = max(0, s - pre_len)
        b1 = s
        if b1 > b0:
            pre_speed.append(float(np.nanmean(run_interp[b0:b1])))
        else:
            pre_speed.append(np.nan)

    return (
        np.asarray(stim_speed, dtype=FLOAT_NP),
        np.asarray(pre_speed, dtype=FLOAT_NP),
        run_interp,
    )

def get_available_stimuli(ds):
    stim_names = set()
    if hasattr(ds, "list_stimuli"):
        try:
            listed = ds.list_stimuli()
            if listed is not None:
                stim_names.update([str(x) for x in listed])
        except Exception:
            pass
    stim_names.update(CFG["CANDIDATE_STIMULI"])
    return sorted(stim_names)

def fetch_stimulus_table_safe(ds, stim_name: str):
    try:
        stim = ds.get_stimulus_table(stim_name).copy()
        stim = stim.reset_index(drop=False).rename(columns={"index": "stimulus_presentation_id"})
        return stim
    except Exception:
        return None

def fetch_stimulus_template_safe(ds, stim_name: str):
    try:
        template = ds.get_stimulus_template(stim_name)
        if template is None:
            return None
        return np.asarray(template)
    except Exception:
        return None

def augment_stimulus_table_with_time_and_running(stim_df, t_ophys, run_time, run_speed):
    stim_df = stim_df.copy()

    if "start" in stim_df.columns:
        start_frames = stim_df["start"].to_numpy(dtype=np.int64)
        stim_df["start_time_s"] = t_ophys[np.clip(start_frames, 0, len(t_ophys) - 1)]
    else:
        start_frames = None

    if "end" in stim_df.columns:
        end_frames = stim_df["end"].to_numpy(dtype=np.int64)
        stim_df["end_time_s"] = t_ophys[np.clip(end_frames, 0, len(t_ophys) - 1)]
    else:
        end_frames = None

    if start_frames is not None and end_frames is not None:
        stim_speed, pre_speed, _ = compute_trial_running_speed(
            t_ophys, run_time, run_speed, start_frames, end_frames
        )
        stim_df["running_speed_mean_cm_per_s"] = stim_speed.astype(np.float64)
        stim_df["running_speed_pre_cm_per_s"] = pre_speed.astype(np.float64)

    return stim_df

def save_template_to_global_bank(stim_name: str, template: np.ndarray):
    sha1 = sha1_of_array(template)
    final_path = bank_template_path(stim_name, sha1)

    if not final_path.exists():
        tmp_path = BANK_DIR / f".tmp_{stim_name}_{uuid.uuid4().hex}.npz"
        np.savez_compressed(
            tmp_path,
            template=template,
            stimulus_name=np.array([stim_name], dtype=object),
            sha1=np.array([sha1], dtype=object),
        )
        os.replace(tmp_path, final_path)

    return {
        "template_hash": sha1,
        "template_bank_relpath": str(final_path.relative_to(ROOT)),
        "template_shape": str(tuple(template.shape)),
        "template_ndim": int(template.ndim),
        "template_dtype": str(template.dtype),
    }

In [ ]:
# %%
# 07 — Neural summary helpers
def build_trial_response_matrices(dff_traces, start_frames, end_frames):
    n_cells, _ = dff_traces.shape
    n_trials = len(start_frames)

    baseline_mean = np.zeros((n_trials, n_cells), dtype=FLOAT_NP)
    response_mean = np.zeros((n_trials, n_cells), dtype=FLOAT_NP)
    delta_mean = np.zeros((n_trials, n_cells), dtype=FLOAT_NP)

    for i, (s, e) in enumerate(zip(start_frames, end_frames)):
        s = int(s)
        e = int(max(e, s + 1))

        resp = dff_traces[:, s:e].mean(axis=1)

        pre_len = e - s
        b0 = max(0, s - pre_len)
        b1 = s
        if b1 > b0:
            base = dff_traces[:, b0:b1].mean(axis=1)
        else:
            base = np.zeros_like(resp)

        baseline_mean[i] = base.astype(FLOAT_NP)
        response_mean[i] = resp.astype(FLOAT_NP)
        delta_mean[i] = (resp - base).astype(FLOAT_NP)

    return baseline_mean, response_mean, delta_mean

def clean_static_gratings_table(stim_df):
    stim = stim_df.copy()

    required_cols = ["start", "end", "orientation", "spatial_frequency", "phase"]
    for c in required_cols:
        if c not in stim.columns:
            raise KeyError(f"Missing expected static-gratings column: {c}")

    if "blank_sweep" in stim.columns:
        stim["is_blank"] = stim["blank_sweep"].fillna(False).astype(bool)
    else:
        stim["is_blank"] = stim[["orientation", "spatial_frequency", "phase"]].isna().any(axis=1)

    stim = stim.sort_values("start").reset_index(drop=True)
    stim_nonblank = stim[~stim["is_blank"]].copy().reset_index(drop=True)
    return stim_nonblank, stim

def build_condition_index(stim_nonblank):
    ori_vals = np.sort(stim_nonblank["orientation"].dropna().unique()).astype(np.float64)
    sf_vals = np.sort(stim_nonblank["spatial_frequency"].dropna().unique()).astype(np.float64)
    phase_vals = np.sort(stim_nonblank["phase"].dropna().unique()).astype(np.float64)

    ori_map = {v: i for i, v in enumerate(ori_vals)}
    sf_map = {v: i for i, v in enumerate(sf_vals)}
    phase_map = {v: i for i, v in enumerate(phase_vals)}

    cond_idx = np.stack([
        stim_nonblank["orientation"].map(ori_map).to_numpy(dtype=np.int32),
        stim_nonblank["spatial_frequency"].map(sf_map).to_numpy(dtype=np.int32),
        stim_nonblank["phase"].map(phase_map).to_numpy(dtype=np.int32),
    ], axis=1)

    O, S, P = len(ori_vals), len(sf_vals), len(phase_vals)
    cond_flat = cond_idx[:, 0] * (S * P) + cond_idx[:, 1] * P + cond_idx[:, 2]

    return ori_vals, sf_vals, phase_vals, cond_idx, cond_flat.astype(np.int32)

def compute_condition_means_and_counts(X, cond_idx, ori_vals, sf_vals, phase_vals):
    O, S, P = len(ori_vals), len(sf_vals), len(phase_vals)
    C = X.shape[1]

    mu = np.full((O, S, P, C), np.nan, dtype=np.float64)
    counts = np.zeros((O, S, P), dtype=np.int32)

    for oi in range(O):
        for si in range(S):
            for pi in range(P):
                mask = (
                    (cond_idx[:, 0] == oi)
                    & (cond_idx[:, 1] == si)
                    & (cond_idx[:, 2] == pi)
                )
                Xi = X[mask]
                counts[oi, si, pi] = Xi.shape[0]
                if Xi.shape[0] > 0:
                    mu[oi, si, pi] = Xi.mean(axis=0)

    return mu.astype(FLOAT_NP), counts

def compute_pooled_cov_delta(delta_trials, cond_idx):
    residuals = []
    for c in np.unique(cond_idx, axis=0):
        mask = np.all(cond_idx == c[None, :], axis=1)
        Xi = delta_trials[mask].astype(np.float64)
        if Xi.shape[0] < 2:
            continue
        residuals.append(Xi - Xi.mean(axis=0, keepdims=True))

    if len(residuals) == 0:
        C = delta_trials.shape[1]
        return np.eye(C, dtype=FLOAT_NP), np.nan

    residuals = np.concatenate(residuals, axis=0)
    lw = LedoitWolf(assume_centered=True)
    lw.fit(residuals)
    return lw.covariance_.astype(FLOAT_NP), float(lw.shrinkage_)

In [ ]:
# %%
# 08 — Completeness helpers
def h5_is_complete(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        with h5py.File(path, "r") as f:
            if "meta" not in f:
                return False
            attrs = f["meta"].attrs
            if str(attrs.get("schema_version", "")) != CFG["SCHEMA_VERSION"]:
                return False
            if bool(attrs.get("preproc_complete", False)) is not True:
                return False
            required_groups = ["meta", "full", "stimuli", "stimulus_refs", "trial_summaries"]
            if not all(g in f for g in required_groups):
                return False
            if "timestamps" not in f["full"]:
                return False
            if "cell_specimen_ids" not in f["full"]:
                return False
        return True
    except Exception:
        return False

def safe_meta_value(x):
    if x is None:
        return "None"
    return x

In [ ]:
# %%
# 09 — One-experiment worker
def preprocess_one_experiment(row_dict):
    exp_id = int(row_dict["id"])
    out_dir = exp_dir(exp_id)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_h5 = store_path(exp_id)

    try:
        if h5_is_complete(out_h5):
            return {"id": exp_id, "status": "exists", "path_h5": str(out_h5)}

        ds = boc.get_ophys_experiment_data(exp_id)

        # core traces
        t_dff, dff = ds.get_dff_traces()
        t_dff = np.asarray(t_dff, dtype=np.float64).ravel()
        dff = np.asarray(dff, dtype=FLOAT_NP)
        cell_specimen_ids = np.asarray(ds.get_cell_specimen_ids(), dtype=np.int64)

        corrected_t = None
        corrected = None
        if CFG["SAVE_FULL_CORRECTED_FLUORESCENCE"]:
            corrected_t, corrected = ds.get_corrected_fluorescence_traces()
            corrected_t = np.asarray(corrected_t, dtype=np.float64).ravel()
            corrected = np.asarray(corrected, dtype=FLOAT_NP)

        roi_masks = None
        if CFG["SAVE_ROI_MASKS"]:
            roi_masks = np.asarray(ds.get_roi_mask_array(), dtype=np.uint8)

        # authoritative running speed
        run_speed, run_time = authoritative_running_speed_from_sdk(ds)
        run_interp = np.interp(t_dff, run_time, run_speed).astype(FLOAT_NP)

        # stimuli
        stim_names = get_available_stimuli(ds)
        stimulus_manifest_rows = []
        stimulus_ref_rows = []
        all_tables = {}
        trial_summaries = {}
        static_condition_summary = None

        for stim_name in stim_names:
            stim_df = fetch_stimulus_table_safe(ds, stim_name)

            if stim_df is None:
                stimulus_manifest_rows.append({
                    "stimulus_name": stim_name,
                    "has_table": False,
                    "n_presentations": 0,
                    "has_template": False,
                    "template_hash": "",
                    "template_bank_relpath": "",
                    "template_shape": "",
                    "template_ndim": np.nan,
                    "template_dtype": "",
                    "summarized": False,
                    "summary_skip_reason": "no_table",
                })
                stimulus_ref_rows.append({
                    "stimulus_name": stim_name,
                    "has_template": False,
                    "template_hash": "",
                    "template_bank_relpath": "",
                    "template_shape": "",
                    "template_ndim": np.nan,
                    "template_dtype": "",
                })
                continue

            stim_df = augment_stimulus_table_with_time_and_running(
                stim_df, t_dff, run_time, run_speed
            )
            all_tables[stim_name] = stim_df

            template_ref = {
                "stimulus_name": stim_name,
                "has_template": False,
                "template_hash": "",
                "template_bank_relpath": "",
                "template_shape": "",
                "template_ndim": np.nan,
                "template_dtype": "",
            }

            if CFG["SAVE_STIMULUS_TEMPLATES_GLOBAL"]:
                template = fetch_stimulus_template_safe(ds, stim_name)
                if template is not None:
                    bank_info = save_template_to_global_bank(stim_name, template)
                    template_ref.update({
                        "has_template": True,
                        **bank_info,
                    })

            summarized = False
            summary_skip_reason = "not_selected"

            if stim_name in CFG["SUMMARY_STIMULI"]:
                if not {"start", "end"}.issubset(stim_df.columns):
                    summary_skip_reason = "missing_start_or_end"
                elif len(stim_df) > CFG["MAX_PRESENTATIONS_FOR_SUMMARY"]:
                    summary_skip_reason = "too_many_presentations"
                else:
                    try:
                        start_frames = stim_df["start"].to_numpy(dtype=np.int64)
                        end_frames = stim_df["end"].to_numpy(dtype=np.int64)

                        baseline_mean, response_mean, delta_mean = build_trial_response_matrices(
                            dff, start_frames, end_frames
                        )
                        trial_run_speed, trial_pre_speed, _ = compute_trial_running_speed(
                            t_dff, run_time, run_speed, start_frames, end_frames
                        )

                        trial_summaries[stim_name] = {
                            "baseline_mean": baseline_mean,
                            "response_mean": response_mean,
                            "delta_mean": delta_mean,
                            "trial_run_speed": trial_run_speed,
                            "trial_pre_speed": trial_pre_speed,
                            "n_presentations": len(stim_df),
                        }

                        summarized = True
                        summary_skip_reason = "ok"

                        if stim_name == "static_gratings" and CFG["COMPUTE_STATIC_CONDITION_SUMMARY"]:
                            stim_nonblank, stim_full = clean_static_gratings_table(stim_df)

                            if len(stim_nonblank) > 0:
                                start_frames_nb = stim_nonblank["start"].to_numpy(dtype=np.int64)
                                end_frames_nb = stim_nonblank["end"].to_numpy(dtype=np.int64)

                                baseline_mean_nb, response_mean_nb, delta_mean_nb = build_trial_response_matrices(
                                    dff, start_frames_nb, end_frames_nb
                                )
                                trial_run_speed_nb, trial_pre_speed_nb, _ = compute_trial_running_speed(
                                    t_dff, run_time, run_speed, start_frames_nb, end_frames_nb
                                )

                                ori_vals, sf_vals, phase_vals, cond_idx, cond_flat = build_condition_index(stim_nonblank)

                                cond_mean_delta, cond_count = compute_condition_means_and_counts(
                                    delta_mean_nb, cond_idx, ori_vals, sf_vals, phase_vals
                                )
                                cond_mean_response, _ = compute_condition_means_and_counts(
                                    response_mean_nb, cond_idx, ori_vals, sf_vals, phase_vals
                                )
                                cond_mean_baseline, _ = compute_condition_means_and_counts(
                                    baseline_mean_nb, cond_idx, ori_vals, sf_vals, phase_vals
                                )

                                cov_delta, shrinkage = compute_pooled_cov_delta(delta_mean_nb, cond_idx)

                                static_condition_summary = {
                                    "stim_nonblank": stim_nonblank,
                                    "stim_full": stim_full,
                                    "baseline_mean": baseline_mean_nb,
                                    "response_mean": response_mean_nb,
                                    "delta_mean": delta_mean_nb,
                                    "trial_run_speed": trial_run_speed_nb,
                                    "trial_pre_speed": trial_pre_speed_nb,
                                    "ori_vals": ori_vals,
                                    "sf_vals": sf_vals,
                                    "phase_vals": phase_vals,
                                    "cond_idx": cond_idx,
                                    "cond_flat": cond_flat,
                                    "cond_mean_baseline": cond_mean_baseline,
                                    "cond_mean_response": cond_mean_response,
                                    "cond_mean_delta": cond_mean_delta,
                                    "cond_count": cond_count,
                                    "cov_delta": cov_delta,
                                    "cov_shrinkage": shrinkage,
                                }
                    except Exception:
                        summarized = False
                        summary_skip_reason = "summary_error"

            stimulus_manifest_rows.append({
                "stimulus_name": stim_name,
                "has_table": True,
                "n_presentations": int(len(stim_df)),
                "has_template": bool(template_ref["has_template"]),
                "template_hash": template_ref["template_hash"],
                "template_bank_relpath": template_ref["template_bank_relpath"],
                "template_shape": template_ref["template_shape"],
                "template_ndim": template_ref["template_ndim"],
                "template_dtype": template_ref["template_dtype"],
                "summarized": summarized,
                "summary_skip_reason": summary_skip_reason,
            })
            stimulus_ref_rows.append(template_ref)

        stimulus_manifest = pd.DataFrame(stimulus_manifest_rows).sort_values("stimulus_name").reset_index(drop=True)
        stimulus_refs = pd.DataFrame(stimulus_ref_rows).sort_values("stimulus_name").reset_index(drop=True)

        has_static = static_condition_summary is not None
        meta_payload = {
            "id": exp_id,
            "path_h5": str(out_h5),
            "schema_version": CFG["SCHEMA_VERSION"],
            "notebook_version": CFG["NOTEBOOK_VERSION"],
            "preproc_complete": False,

            "targeted_structure": row_dict.get("targeted_structure", None),
            "imaging_depth": row_dict.get("imaging_depth", None),
            "cre_line": row_dict.get("cre_line", None),
            "experiment_container_id": row_dict.get("experiment_container_id", None),
            "session_type": row_dict.get("session_type", None),
            "donor_name": row_dict.get("donor_name", None),

            "is_excitatory": bool(is_excitatory(row_dict.get("cre_line", ""))),
            "visp_layer_label": canonical_visp_layer_label(
                row_dict.get("cre_line", ""),
                row_dict.get("imaging_depth", None),
            ),

            "n_cells": int(dff.shape[0]),
            "n_time": int(dff.shape[1]),
            "n_available_stimuli": int(stimulus_manifest["has_table"].sum()),

            "has_static_gratings": bool("static_gratings" in all_tables),
            "has_drifting_gratings": bool("drifting_gratings" in all_tables),
            "has_locally_sparse_noise": bool("locally_sparse_noise" in all_tables),
            "has_locally_sparse_noise_4deg": bool("locally_sparse_noise_4deg" in all_tables),
            "has_locally_sparse_noise_8deg": bool("locally_sparse_noise_8deg" in all_tables),

            "has_any_lsn": bool(
                "locally_sparse_noise" in all_tables
                or "locally_sparse_noise_4deg" in all_tables
                or "locally_sparse_noise_8deg" in all_tables
            ),

            "has_static_condition_summary": bool(has_static),
            "save_full_dff": bool(CFG["SAVE_FULL_DFF"]),
            "running_speed_from_sdk": True,
        }

        if has_static:
            meta_payload.update({
                "n_trials_nonblank_static": int(len(static_condition_summary["stim_nonblank"])),
                "n_conditions_static": int(len(np.unique(static_condition_summary["cond_flat"]))),
                "n_ori_static": int(len(static_condition_summary["ori_vals"])),
                "n_sf_static": int(len(static_condition_summary["sf_vals"])),
                "n_phase_static": int(len(static_condition_summary["phase_vals"])),
            })

        with open(meta_json_path(exp_id), "w") as f:
            json.dump({k: safe_jsonable(v) for k, v in meta_payload.items()}, f, indent=2)

        with h5py.File(out_h5, "w") as f:
            g_meta = f.create_group("meta")
            for k, v in meta_payload.items():
                g_meta.attrs[k] = safe_meta_value(v)

            g_full = f.create_group("full")
            write_h5_dataset(g_full, "timestamps", t_dff.astype(np.float64))
            write_h5_dataset(g_full, "cell_specimen_ids", cell_specimen_ids)

            if CFG["SAVE_FULL_DFF"]:
                write_h5_dataset(g_full, "dff", dff)

            if CFG["SAVE_FULL_CORRECTED_FLUORESCENCE"] and corrected is not None:
                write_h5_dataset(g_full, "corrected_fluorescence_timestamps", corrected_t.astype(np.float64))
                write_h5_dataset(g_full, "corrected_fluorescence", corrected)

            if CFG["SAVE_RUNNING_FULL"]:
                write_h5_dataset(g_full, "running_speed_time", run_time.astype(np.float64))
                write_h5_dataset(g_full, "running_speed_cm_per_s", run_speed.astype(FLOAT_NP))
                write_h5_dataset(g_full, "running_speed_on_ophys_grid_cm_per_s", run_interp.astype(FLOAT_NP))

            if roi_masks is not None:
                write_h5_dataset(g_full, "roi_masks", roi_masks)

            g_stim = f.create_group("stimuli")
            write_dataframe_to_h5(g_stim, "manifest", stimulus_manifest)
            if CFG["SAVE_STIMULUS_TABLES_IN_H5"]:
                for stim_name, stim_df in all_tables.items():
                    write_dataframe_to_h5(g_stim, stim_name, stim_df)

            g_refs = f.create_group("stimulus_refs")
            write_dataframe_to_h5(g_refs, "manifest", stimulus_refs)

            g_trial = f.create_group("trial_summaries")
            for stim_name, payload in trial_summaries.items():
                g_one = g_trial.create_group(stim_name)
                g_one.attrs["n_presentations"] = int(payload["n_presentations"])
                write_h5_dataset(g_one, "baseline_mean_dff", payload["baseline_mean"])
                write_h5_dataset(g_one, "response_mean_dff", payload["response_mean"])
                write_h5_dataset(g_one, "delta_mean_dff", payload["delta_mean"])
                write_h5_dataset(g_one, "running_speed_mean_cm_per_s", payload["trial_run_speed"])
                write_h5_dataset(g_one, "running_speed_pre_cm_per_s", payload["trial_pre_speed"])

            if has_static:
                g_static = f.create_group("static_gratings_summary")
                write_dataframe_to_h5(g_static, "stim_nonblank_table", static_condition_summary["stim_nonblank"])

                g_trials = g_static.create_group("trials")
                write_h5_dataset(g_trials, "baseline_mean_dff", static_condition_summary["baseline_mean"])
                write_h5_dataset(g_trials, "response_mean_dff", static_condition_summary["response_mean"])
                write_h5_dataset(g_trials, "delta_mean_dff", static_condition_summary["delta_mean"])
                write_h5_dataset(g_trials, "running_speed_mean_cm_per_s", static_condition_summary["trial_run_speed"])
                write_h5_dataset(g_trials, "running_speed_pre_cm_per_s", static_condition_summary["trial_pre_speed"])

                g_design = g_static.create_group("design")
                write_h5_dataset(g_design, "orientation_values_deg", static_condition_summary["ori_vals"].astype(np.float64))
                write_h5_dataset(g_design, "spatial_frequency_values_cpd", static_condition_summary["sf_vals"].astype(np.float64))
                write_h5_dataset(g_design, "phase_values", static_condition_summary["phase_vals"].astype(np.float64))
                write_h5_dataset(g_design, "cond_idx", static_condition_summary["cond_idx"].astype(np.int32))
                write_h5_dataset(g_design, "cond_flat", static_condition_summary["cond_flat"].astype(np.int32))

                g_sum = g_static.create_group("summary")
                write_h5_dataset(g_sum, "cond_mean_baseline", static_condition_summary["cond_mean_baseline"])
                write_h5_dataset(g_sum, "cond_mean_response", static_condition_summary["cond_mean_response"])
                write_h5_dataset(g_sum, "cond_mean_delta", static_condition_summary["cond_mean_delta"])
                write_h5_dataset(g_sum, "cond_count", static_condition_summary["cond_count"].astype(np.int32))
                if CFG["COMPUTE_POOLED_COV_DELTA"]:
                    write_h5_dataset(g_sum, "cov_delta", static_condition_summary["cov_delta"].astype(FLOAT_NP))
                    g_sum.attrs["cov_delta_ledoitwolf_shrinkage"] = static_condition_summary["cov_shrinkage"]

            f["meta"].attrs["preproc_complete"] = True

        return {
            "id": exp_id,
            "status": "ok",
            "path_h5": str(out_h5),
            "targeted_structure": row_dict.get("targeted_structure", None),
            "session_type": row_dict.get("session_type", None),
            "n_cells": int(dff.shape[0]),
            "n_available_stimuli": int(stimulus_manifest["has_table"].sum()),
            "has_static_condition_summary": bool(has_static),
        }

    except Exception as e:
        return {
            "id": exp_id,
            "status": "error",
            "error": repr(e),
            "traceback": traceback.format_exc(),
        }

In [ ]:
# %%
# 10 — Parallel preprocessing run
task_df = meta_df.copy()
if CFG["MAX_EXPERIMENTS"] is not None:
    task_df = task_df.head(int(CFG["MAX_EXPERIMENTS"])).copy()

task_rows = task_df.to_dict(orient="records")

print("Experiments scheduled:", len(task_rows))
print("Parallel workers:", CFG["N_WORKERS"])

results = Parallel(
    n_jobs=CFG["N_WORKERS"],
    backend="loky",
    batch_size=1,
    verbose=10,
)(
    delayed(preprocess_one_experiment)(row_dict)
    for row_dict in task_rows
)

preproc_log = pd.DataFrame(results)
preproc_log.to_csv(LOG_DIR / "preprocessing_log.csv", index=False)

print("Status counts:")
display(preproc_log["status"].value_counts(dropna=False))

errors_df = preproc_log[preproc_log["status"] == "error"].copy()
if len(errors_df):
    print("Example errors:")
    display(errors_df[["id", "error"]].head(20))
else:
    print("No preprocessing errors.")

In [ ]:
# %%
# 11 — Build reusable index CSV
ok_df = preproc_log[preproc_log["status"].isin(["ok", "exists"])].copy()
index_df = ok_df[["id", "path_h5"]].merge(meta_df, on="id", how="left")

extra_rows = []
for row in index_df.itertuples(index=False):
    with h5py.File(row.path_h5, "r") as f:
        attrs = dict(f["meta"].attrs)
    extra_rows.append({
        "id": int(row.id),
        "n_cells": int(attrs["n_cells"]),
        "n_time": int(attrs["n_time"]),
        "n_available_stimuli": int(attrs["n_available_stimuli"]),
        "has_static_gratings": bool(attrs["has_static_gratings"]),
        "has_drifting_gratings": bool(attrs["has_drifting_gratings"]),
        "has_locally_sparse_noise": bool(attrs["has_locally_sparse_noise"]),
        "has_locally_sparse_noise_4deg": bool(attrs["has_locally_sparse_noise_4deg"]),
        "has_locally_sparse_noise_8deg": bool(attrs["has_locally_sparse_noise_8deg"]),
        "has_any_lsn": bool(attrs["has_any_lsn"]),
        "has_static_condition_summary": bool(attrs["has_static_condition_summary"]),
        "n_trials_nonblank_static": int(attrs["n_trials_nonblank_static"]) if "n_trials_nonblank_static" in attrs else 0,
        "n_conditions_static": int(attrs["n_conditions_static"]) if "n_conditions_static" in attrs else 0,
        "n_ori_static": int(attrs["n_ori_static"]) if "n_ori_static" in attrs else 0,
        "n_sf_static": int(attrs["n_sf_static"]) if "n_sf_static" in attrs else 0,
        "n_phase_static": int(attrs["n_phase_static"]) if "n_phase_static" in attrs else 0,
        "is_excitatory": bool(attrs["is_excitatory"]),
        "visp_layer_label": str(attrs["visp_layer_label"]),
        "save_full_dff": bool(attrs["save_full_dff"]),
        "schema_version": str(attrs["schema_version"]),
        "notebook_version": str(attrs["notebook_version"]),
        "preproc_complete": bool(attrs["preproc_complete"]),
    })

extra_df = pd.DataFrame(extra_rows)
index_df = index_df.merge(extra_df, on="id", how="left")
preferred_cols = [
    "id",
    "path_h5",
    "experiment_container_id",
    "session_type",
    "targeted_structure",
    "imaging_depth",
    "cre_line",
    "donor_name",
    "is_excitatory",
    "visp_layer_label",
    "n_cells",
    "n_time",
    "n_available_stimuli",
    "has_static_gratings",
    "has_drifting_gratings",
    "has_locally_sparse_noise",
    "has_locally_sparse_noise_4deg",
    "has_locally_sparse_noise_8deg",
    "has_any_lsn",
    "has_static_condition_summary",
    "n_trials_nonblank_static",
    "n_conditions_static",
    "n_ori_static",
    "n_sf_static",
    "n_phase_static",
    "save_full_dff",
    "schema_version",
    "notebook_version",
    "preproc_complete",
]
preferred_cols = [c for c in preferred_cols if c in index_df.columns]

index_df = index_df[preferred_cols].sort_values(
    ["targeted_structure", "imaging_depth", "id"]
).reset_index(drop=True)

index_df.to_csv(INDEX_DIR / "preprocessed_index.csv", index=False)

print("Saved:", INDEX_DIR / "preprocessed_index.csv")
display(index_df.head())

In [ ]:
# %%
# 12 — Build global stimulus-bank index and manifest
bank_rows = []
for p in sorted(BANK_DIR.glob("*.npz")):
    try:
        z = np.load(p, allow_pickle=True)
        template = z["template"]
        stim_name = str(z["stimulus_name"][0])
        sha1 = str(z["sha1"][0])

        bank_rows.append({
            "stimulus_name": stim_name,
            "template_hash": sha1,
            "template_shape": str(tuple(template.shape)),
            "template_ndim": int(template.ndim),
            "template_dtype": str(template.dtype),
            "size_mb": p.stat().st_size / (1024 ** 2),
            "path": str(p),
        })
    except Exception as e:
        bank_rows.append({
            "stimulus_name": "ERROR",
            "template_hash": "",
            "template_shape": "",
            "template_ndim": np.nan,
            "template_dtype": "",
            "size_mb": np.nan,
            "path": str(p),
            "error": repr(e),
        })

bank_df = pd.DataFrame(bank_rows)
bank_df.to_csv(INDEX_DIR / "stimulus_bank_index.csv", index=False)

manifest = {
    "root": str(ROOT),
    "index_csv": str(INDEX_DIR / "preprocessed_index.csv"),
    "stimulus_bank_index_csv": str(INDEX_DIR / "stimulus_bank_index.csv"),
    "n_preprocessed_experiments": int(len(index_df)),
    "config": CFG,
}

with open(TRANSPORT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved:", INDEX_DIR / "stimulus_bank_index.csv")
print("Saved:", TRANSPORT_DIR / "manifest.json")
display(bank_df.head())

In [ ]:
# %%
# 13 — QC summaries
print("Preprocessed experiments:", len(index_df))

print("\nBy area:")
display(
    index_df.groupby("targeted_structure")
    .size()
    .rename("n_experiments")
    .reset_index()
    .sort_values("n_experiments", ascending=False)
)

print("\nBy session type:")
display(
    index_df.groupby("session_type")
    .size()
    .rename("n_experiments")
    .reset_index()
    .sort_values("n_experiments", ascending=False)
)

stim_cols = [
    "has_static_gratings",
    "has_drifting_gratings",
    "has_locally_sparse_noise",
    "has_locally_sparse_noise_4deg",
    "has_locally_sparse_noise_8deg",
    "has_any_lsn",
]

print("\nStimulus availability counts:")
print(index_df[stim_cols].sum().to_string())

static_df = index_df[index_df["has_static_condition_summary"]].copy()
print("\nStatic summary-ready experiments:", len(static_df))
if len(static_df):
    print(
        static_df[
            ["n_trials_nonblank_static", "n_conditions_static", "n_ori_static", "n_sf_static", "n_phase_static"]
        ].describe().to_string()
    )


In [ ]:
# %%
# 14 — Running-speed sanity checks
speed_rows = []
for row in index_df.itertuples(index=False):
    with h5py.File(row.path_h5, "r") as f:
        run_interp = f["full/running_speed_on_ophys_grid_cm_per_s"][:].astype(np.float64)

        if "static_gratings_summary" in f:
            trial_speed = f["static_gratings_summary/trials/running_speed_mean_cm_per_s"][:].astype(np.float64)
            n_still_025 = int(np.sum(trial_speed <= 0.25))
            n_run_1 = int(np.sum(trial_speed >= 1.0))
            trial_min = float(np.nanmin(trial_speed))
            trial_median = float(np.nanmedian(trial_speed))
            trial_max = float(np.nanmax(trial_speed))
        else:
            n_still_025 = np.nan
            n_run_1 = np.nan
            trial_min = np.nan
            trial_median = np.nan
            trial_max = np.nan

    speed_rows.append({
        "id": int(row.id),
        "run_interp_min": float(np.nanmin(run_interp)),
        "run_interp_median": float(np.nanmedian(run_interp)),
        "run_interp_max": float(np.nanmax(run_interp)),
        "trial_min_static": trial_min,
        "trial_median_static": trial_median,
        "trial_max_static": trial_max,
        "n_still_025_static": n_still_025,
        "n_run_1_static": n_run_1,
    })

speed_df = pd.DataFrame(speed_rows)
speed_df.to_csv(LOG_DIR / "running_speed_sanity_summary.csv", index=False)

display(speed_df.head(20))
print(speed_df.describe().to_string())

In [ ]:

# %%
# 16 — Inspect one stored HDF5 file
sample_id = int(index_df.iloc[0]["id"])
sample_path = index_df.iloc[0]["path_h5"]

print("Sample experiment id:", sample_id)
print("Sample path:", sample_path)

with h5py.File(sample_path, "r") as f:
    print("\nGroups:", list(f.keys()))
    print("\n/full keys:", list(f["full"].keys()))
    print("\n/stimuli keys:", list(f["stimuli"].keys())[:12], "..." if len(f["stimuli"].keys()) > 12 else "")
    print("\n/stimulus_refs keys:", list(f["stimulus_refs"].keys()))
    print("\n/trial_summaries keys:", list(f["trial_summaries"].keys()))
    if "static_gratings_summary" in f:
        print("\n/static_gratings_summary keys:", list(f["static_gratings_summary"].keys()))

    print("\nMeta attrs:")
    for k, v in dict(f["meta"].attrs).items():
        print(f"{k}: {v}")

In [ ]:
# %%
# 17 — Optional transport tarball
MAKE_TARBALL = False
TARBALL_PATH = TRANSPORT_DIR / "allen_preproc_gratings_lsn_v1_portable.tar"

if MAKE_TARBALL:
    import tarfile

    with tarfile.open(TARBALL_PATH, "w") as tar:
        tar.add(INDEX_DIR / "preprocessed_index.csv", arcname="indexes/preprocessed_index.csv")
        tar.add(INDEX_DIR / "stimulus_bank_index.csv", arcname="indexes/stimulus_bank_index.csv")
        tar.add(TRANSPORT_DIR / "manifest.json", arcname="transport/manifest.json")

        for row in index_df.itertuples(index=False):
            folder = exp_dir(int(row.id))
            tar.add(folder, arcname=f"experiments/{folder.name}")

        tar.add(BANK_DIR, arcname="stimulus_bank")

    print("Created:", TARBALL_PATH)
else:
    print("Skipping tarball creation.")